# படி 07 - திட்டமிடல் வடிவமைப்பு மாதிரி

இந்த நோட்புக் மைக்ரோசாஃப்ட் ஏஜென்ட் கட்டமைப்பைக் கொண்டு AI ஏஜெண்டுகளுக்கான **திட்டமிடல் வடிவமைப்பு மாதிரியை** காட்டுகிறது.
நீங்கள் கடினமான பயணம் கோரிக்கைகளை கட்டமைக்கப்பட்ட துணைபணிகளாக உடைக்க எவ்வாறு செய்வது, அவற்றை நிபுணர் ஏஜெண்டுகளுக்கு ஒதுக்குவது,
மற்றும் உரிய திட்டத்தை செயல்படுத்துவது — அனைத்தும் Pydantic மாதிரிகளால் இயக்கப்படும் கட்டமைக்கப்பட்ட வெளியீட்டை பயன்படுத்தி கற்றுக்கொள்ளப்போகிறீர்கள்.


## நிறுவல்


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## பணி உடைத்தல்

பணி உடைத்தல் என்பது திட்டமிடும் வடிவமைப்பு முறைமையின் மையக்கருத்தாகும். ஒரு சிக்கலான கோரிக்கையை ஒரு ஒரே முகவரிடம் முழுமையாக கையாளக் கேட்கும் பதிலாக,
பிரச்சினையை சிறிய, நன்கு வரையறுக்கப்பட்ட **துணைப்பணிகள்** ஆக பிரிக்கிறோம்.
ஒவ்வொரு துணைப்பணியும் ஒரு நிபுணர் முகவரிடம் (எ.கா., விமானங்கள், ஹோட்டல்கள், நடவடிக்கைகள்) நியமிக்கப்பட்டு, தெளிவான
முன்னுரிமைகள் மற்றும் சார்பு வரிசையுடன் வழங்கப்படுகிறது.

இந்த அணுகுமுறை பல நன்மைகளை வழங்குகிறது:
- **தெளிவு**: ஒவ்வொரு துணைப்பணிக்கும் ஒரே பொறுப்பு உள்ளது.
- **சமயோஜிதம்**: சுதந்திரமான துணைப்பணிகள் ஒரே நேரத்தில் இயங்க முடியும்.
- **நம்பகத்தன்மை**: தோல்விகள் தனிப்பட்ட துணைப்பணிகளுக்கு மட்டுமே மட்டுப்படுத்தப்படுகின்றன.
- **பட்ஜெட் கண்காணிப்பு**: செலவுகள் ஒவ்வொரு துணைப்பணிக்கும் கணக்கிடப்பட்டு, கூடியமாகக் கணக்கிடப்படுகின்றன.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## கட்டமைக்கப்பட்ட வெளியீட்டுடன் ஒரு திட்டமிடும் முகவரியை உருவாக்குதல்

திட்டமிடும் முகவர் ஒரு **முன்னணி மேசை ஒருங்கிணைப்பாளராக** செயல்படுகிறது. ஒரு உயர்தர பயண கோரிக்கையை வழங்கியபோது,
அது ஒரு கட்டமைக்கப்பட்ட `TravelPlan`-ஐ உருவாக்குகிறது — கோரிக்கையை துணைக்கார்களாக பிரித்து, முன்முயற்சிகளை அமைத்து,
மற்றும் தாழ்வுகளைக் கண்டறிந்து, ஒரு கான்சியெர்ஜ் அல்லது செயல்படுத்தும் படலம் பணியை மேற்கொள்ள உதவுகிறது.


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## சிறப்பு கருவிகளுடன் ஒரு திட்டத்தை செயல்படுத்துதல்

முன் டெ스크 முகவர் ஒரு கட்டமைக்கப்பட்ட திட்டத்தை உருவாக்கியவுடன், **கான்சியூயர் முகவர்** அதை செயல்படுத்தும்.
ஒவ்வொரு சிறப்பு கருவியும் ஒரு வகை துணைபணி (பயண டிக்கெட்டுகள், ஹோட்டல்கள், நடவடிக்கைகள்) கையாள்கிறது. கான்சியூயர்
திட்டத்தின் துணைபணிகளை சார்ப dependency வரிசையில் வழிநடத்தி ஒவ்வொன்றையும்
பொருத்தமான கருவிக்கு அனுப்புகிறது.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் AI முகவர்கள் க்கான **திட்டமிடல் வடிவமைப்பு முறை** பற்றி கற்கின்றீர்கள்:

1. **பணி உடைமையை இயற்றல்** — ஒரு முன் டெஸ்க் திட்டமிடல் முகவர் ஒரு מורக்கமான பயண கோரிக்கையை Pydantic மாதிரிகள் பயன்படுத்தி
   அமைக்கப்பட்ட துணை பணிகளாக பிரிக்கிறது, ஒவ்வொன்றையும் முன்னுரிமைகள் மற்றும் சார்புகளுடன் நிபுணர் முகவருக்கு
   ஒதுக்குகிறது.
2. **அமைப்புக்குரிய வெளியீடு** — `response_format` இடம் குறித்தல் மூலம் முகவர் ஒரு சரிபார்க்கப்பட்ட
   `TravelPlan` பொருளை விடுவிக்கிறது, இது downstream செயல்முறை நம்பகமாகவும் செய்கிறது.
3. **திட்ட செயல்பாடு** — ஒரு கான்சியெர்ஜ் முகவர் துணை பணிகளை நிபுணர் கருவிகள்
   (`book_flight`, `reserve_hotel`, `book_activity`) பயன்படுத்தி திட்டத்தை நிறைவேற்ற மற்றும் முடிவுகளை அறிக்கை செய்கிறது.

இந்த வடிவமைப்பு *என்ன செய்ய வேண்டும்* (திட்டமிடல்) என்பதை *எப்படி செய்வது* (நிர்வாகம்) இல் இருந்து பிரித்து,
முகவர்களை modular, சோதனைசெய்யக்கூடிய மற்றும் விரிவாக்க எளிதாக்குகிறது.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
